# End-to-end prioritization simulation: official CWE vs. HiCWE-predicted CWE

This notebook closes the gap between the two halves of the paper. The empirical study
(Findings 1–6) shows that **historical CWE exploitation rates rank future exploited CVEs
better than random**. HiCWE predicts CWEs. What has not been shown is whether *predicted*
CWEs preserve that prioritization signal.

The experiment below answers that directly:

1. take CVEs disclosed in 2021–2025 (2023–2025 is the honest held-out window);
2. replace each CVE's official NVD CWE assignment with **HiCWE top-1 / top-3 / top-5**
   predictions;
3. recompute the paper's historical exploitation score from **prior years only** using
   the exact smoothing and max-over-labels rules;
4. rank the CVEs and compare against the **official-CWE**, **EPSS**, **no-CWE**, and
   **random** rankings on PR-AUC, ROC-AUC, P@k, R@k, and NDCG@k.

The key quantity is **signal retention**: how much of the official-CWE ranking performance
survives when the label is predicted rather than assigned by NVD.

> **Run this on a GPU if possible.** The notebook detects CUDA automatically, uses a
> larger batch size on GPU, and caches HiCWE predictions in `experiments/cache/`.
> A full 2021–2025 run is roughly 10–30 minutes on a modern GPU and 1–2 hours on CPU.
> Set `RUN_MODE = "smoke"` in the configuration cell for a quick 500-CVE/year test.


## Running on a GPU

1. Install the CUDA build of PyTorch for your driver, then install the remaining
   requirements (see below).
2. Select a GPU runtime (Colab/Kaggle: *Runtime → Change runtime type → GPU*).
3. Run all cells. The notebook detects CUDA automatically, uses batch size 128, and caches
   HiCWE predictions so a rerun does not re-encode the corpus.
4. On the first run it downloads the public datasets, the CWE hierarchy, the HiCWE
   checkpoint, CISA/ENISA exploitation labels, and EPSS. If `EVAL_YEARS` includes 2021 or
   2022, it also downloads the missing 2020–2022 NVD records (about 10 minutes without an
   NVD API key; set `NVD_API_KEY` in the environment to speed this up).

For the reviewer-critical test alone (the honest 2023–2025 held-out window), set
`EVAL_YEARS = [2023, 2024, 2025]` and `FETCH_MISSING_NVD = False` in the configuration
cell. For a quick end-to-end check, set `RUN_MODE = "smoke"`.


## What is being compared

For each evaluation year `Y`:

| Method | CWE set used for `RankScore(v)` |
| --- | --- |
| `official_all` | all official NVD CWE assignments (paper's empirical-study setup) |
| `official_699` | official assignments restricted to the CWE-699 child space |
| `official_699_primary` | first official CWE-699 child only |
| `hicwe_top1` | HiCWE's single best CWE |
| `hicwe_top3` | top-3 HiCWE candidates, treated as a multi-label set |
| `hicwe_top5` | top-5 HiCWE candidates |
| `no_cwe` | constant global historical exploitation rate (fallback) |
| `random` | constant score (expectation of a random permutation) |
| `epss` | current FIRST EPSS score (retrospective upper bound) |

The **primary comparison** is `hicwe_top1` vs. `official_699`, on the
`official699_matched` cohort. That cohort contains exactly the CVEs for which NVD has a
CWE-699 label, so both methods are evaluated on the same vulnerabilities and the only
difference is whether the label is predicted or official.

### Scoring equations

For a CWE `c` and evaluation year `Y`, using only CVEs published before `Y`:

$$
P(\mathrm{Exploited}\mid c)=
\frac{E_c+\alpha P_0}{N_c+\alpha},
\qquad \alpha=40 .
$$

For a CVE `v` with CWE set `CWE(v)`:

$$
\mathrm{RankScore}(v)=\max_{c\in\mathrm{CWE}(v)} P(\mathrm{Exploited}\mid c).
$$

CVEs with no usable CWE receive the global historical rate `P0`. This is identical to
Eq. (5)–(6) in the paper. P@k, R@k and NDCG@k average over random tie-breaking; the
notebook computes the **exact expectation** analytically instead of Monte-Carlo sampling
1,000 permutations (same metric, no sampling noise).


## Requirements

```bash
# Install the CUDA build of PyTorch for your driver first, e.g.
# pip install torch --index-url https://download.pytorch.org/whl/cu124
pip install -q sentence-transformers scikit-learn pandas numpy matplotlib requests tqdm huggingface_hub
```

The notebook reads local copies of the repo data when they exist, and otherwise downloads:

- `regularpooria/NVD_CVE_699_Dataset` (NVD 2023–2026)
- `regularpooria/CVE_CWE_Software_Mapping_Dataset` (`Global_Dataset.csv`, 1999–2021)
- `regularpooria/HiCWE-MiniLM-L6` or `regularpooria/HiCWE-MiniLM-L12`
- CWE-699 hierarchy from the GitHub repo
- CISA KEV, ENISA EUVD KEV, and the FIRST EPSS snapshot

The 2020–2022 NVD records are not in the published dataset. If `EVAL_YEARS` includes
2021 or 2022, the notebook downloads them from the NVD API on first use and caches
`experiments/cache/nvd_2020_2022_cwe699.csv`. Set `FETCH_MISSING_NVD = False` (or use only
2023–2025) to skip that step.


In [ ]:
import gzip
import hashlib
import json
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "sentence-transformers is not installed. Run:\n"
        "  %pip install -q sentence-transformers scikit-learn pandas numpy matplotlib requests tqdm huggingface_hub"
    ) from exc

try:
    from IPython.display import display
except ImportError:  # allows the code to be run as a plain script
    display = print

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


## Configuration

`MODEL_TAG = "L12"` is the paper's best model. `"L6"` is roughly 3× faster on CPU and
has almost identical flat-child accuracy (0.771 vs. 0.789 at top-1), so it is a good
choice for a quick check. On a GPU, L12 is still cheap.

`SAMPLE_SIZE = None` runs the full panel. For a smoke test, set `RUN_MODE = "smoke"`;
that uses 500 CVEs per year and 2023–2025 only.


In [ ]:
RUN_MODE = "full"          # "full" or "smoke"

EVAL_YEARS = [2021, 2022, 2023, 2024, 2025]
if RUN_MODE == "smoke":
    EVAL_YEARS = [2023, 2024, 2025]
    SAMPLE_SIZE = 500
else:
    SAMPLE_SIZE = None

MODEL_TAG = "L12"          # "L6" or "L12"
ALPHA = 40.0                # paper's smoothing parameter
KS = [50, 100, 250, 500]
RANDOM_SEED = 0
USE_EPSS = True
USE_KEV_CROSS_REF = True         # use scrapers/scraped_data/kev_cross_reference.csv if present
USE_ENISA_KEV = True             # fallback only: public CISA + ENISA labels
EXTRA_EXPLOITATION_CSV = None   # optional CSV with columns cve_id[, date_added, source]
HISTORY_MODE = "publication"    # "publication" (paper) or "knowledge" (leakage-safe)
N_BOOTSTRAP = 500
KEEP_TOPK = 5
FETCH_MISSING_NVD = True

NVD_API_KEY = os.environ.get("NVD_API_KEY", "").strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128 if DEVICE == "cuda" else 32

print(f"Device: {DEVICE}")
print(f"Model: HiCWE-MiniLM-{MODEL_TAG}")
print(f"Evaluation years: {EVAL_YEARS}")
print(f"Sample per year: {SAMPLE_SIZE}")


## Paths and CWE-699 hierarchy

The notebook finds the repository root by looking for `scrapers/scraped_data/CWE_SOFTWARE.json`.
Everything is written under `experiments/cache/` and `experiments/simulation_outputs/`.


In [ ]:
def find_base_dir():
    p = Path.cwd().resolve()
    while True:
        if (p / "scrapers" / "scraped_data" / "CWE_SOFTWARE.json").exists():
            return p
        if p == p.parent:
            return Path.cwd().resolve()
        p = p.parent


BASE_DIR = find_base_dir()
CACHE_DIR = BASE_DIR / "experiments" / "cache"
OUT_DIR = BASE_DIR / "experiments" / "simulation_outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

NVD_LOCAL = BASE_DIR / "dataset" / "nvd_hf_upload" / "NVD_CVE_Dataset.csv"
GLOBAL_LOCAL = BASE_DIR / "dataset" / "hf_upload" / "Global_Dataset.csv"
NVD_HISTORY = CACHE_DIR / "nvd_2020_2022_cwe699.csv"
KEV_CROSS_REF = BASE_DIR / "scrapers" / "scraped_data" / "kev_cross_reference.csv"
CWE_JSON = BASE_DIR / "scrapers" / "scraped_data" / "CWE_SOFTWARE.json"

if not CWE_JSON.exists():
    CWE_JSON = CACHE_DIR / "CWE_SOFTWARE.json"
    if not CWE_JSON.exists():
        url = (
            "https://raw.githubusercontent.com/regularpooria/CWE_CVE_Mapping/"
            "main/scrapers/scraped_data/CWE_SOFTWARE.json"
        )
        print("Downloading CWE-699 hierarchy ...")
        response = requests.get(url, timeout=180)
        response.raise_for_status()
        CWE_JSON.write_bytes(response.content)

print("BASE_DIR:", BASE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def parse_ids(value):
    """Extract numeric CWE ids from list-likes, pipe-separated strings, etc."""
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(v) for v in value if str(v).strip().isdigit()]
    text = str(value)
    if text.lower() in {"nan", "none", ""}:
        return []
    return re.findall(r"\d+", text)


with open(CWE_JSON, encoding="utf-8") as f:
    CWE = json.load(f)

child_lookup = {}
child_to_cats = {}
for cat_id, category in CWE.items():
    cat_id = str(cat_id)
    for child_id, child in category["children"].items():
        child_id = str(child_id)
        child_lookup.setdefault(child_id, child)
        child_to_cats.setdefault(child_id, []).append(cat_id)

CHILD_IDS = sorted(child_lookup, key=lambda x: int(x))
CHILD_ID_SET = set(CHILD_IDS)


def extract_description(child):
    description = child.get("description") or ""
    if child.get("extended_description"):
        description += f"\n{child['extended_description']}"
    return description


CHILD_TEXTS = [extract_description(child_lookup[cid]) for cid in CHILD_IDS]
TAXONOMY_HASH = hashlib.md5("".join(CHILD_TEXTS).encode()).hexdigest()[:8]
print(f"{len(CWE)} categories, {len(CHILD_IDS)} CWE-699 children (taxonomy {TAXONOMY_HASH})")


## Load the CVE panel

The panel combines three sources:

- **NVD 2023–2026** — the published `NVD_CVE_699_Dataset.csv` (local or Hugging Face).
- **NVD 2020–2022** — fetched from the NVD API on first use, because the published dataset
  starts in 2023. These years are needed for the paper's 2021–2025 rolling evaluation.
- **Global 1999–2021** — the older CVE-to-CWE mapping, used as additional history for the
  earliest evaluation years.

NVD rows take precedence over Global rows for the same CVE. The two official label spaces
are kept separately:

- `official_all`: every CWE NVD assigns.
- `official_699`: the subset inside the 399-child CWE-699 Software view, i.e. the space
  HiCWE ranks.


In [ ]:
def download(url, path, timeout=300):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    tmp = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} ...")
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.replace(path)
    return path


def hf_read_csv(repo_id, filename, local_path=None):
    if local_path is not None and Path(local_path).exists():
        return pd.read_csv(local_path, low_memory=False)
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        token=os.getenv("HF_TOKEN"),
    )
    return pd.read_csv(path, low_memory=False)


def fetch_nvd_range(years=(2020, 2021, 2022), out_path=NVD_HISTORY):
    """Download monthly NVD API 2.0 windows and cache the same schema as the HF dataset."""
    out_path = Path(out_path)
    if out_path.exists():
        return out_path
    if not FETCH_MISSING_NVD:
        raise FileNotFoundError(
            f"{out_path} is missing and FETCH_MISSING_NVD=False. "
            "Use EVAL_YEARS=[2023, 2024, 2025] or provide the file."
        )

    from datetime import date, timedelta

    base = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    pause = 0.7 if NVD_API_KEY else 6.5

    def windows(year):
        d = date(year, 1, 1)
        end = date(year, 12, 31)
        while d <= end:
            e = min(d + timedelta(days=110), end)
            yield d, e
            d = e + timedelta(days=1)

    def parse(v):
        cve = v["cve"]
        desc = next(
            (d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"),
            None,
        )
        cwes = []
        for w in cve.get("weaknesses", []):
            for d in w.get("description", []):
                val = d.get("value", "")
                if val.startswith("CWE-"):
                    cwes.append(val)
        cwes = sorted(set(cwes))
        matches = [c for c in cwes if c.split("-", 1)[1] in CHILD_ID_SET]
        cvss = None
        metrics = cve.get("metrics", {})
        for key in ("cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
            if metrics.get(key):
                try:
                    cvss = metrics[key][0]["cvssData"]["baseScore"]
                except Exception:
                    pass
                break
        return {
            "cve_id": cve["id"],
            "published": cve.get("published"),
            "last_modified": cve.get("lastModified"),
            "published_year": int(cve["published"][:4]),
            "description": desc,
            "all_nvd_cwes": "|".join(cwes) if cwes else None,
            "cwe_699_matches": "|".join(matches) if matches else None,
            "number_of_nvd_cwes": len(cwes),
            "number_of_cwe_699_matches": len(matches),
            "cvss_base": cvss,
        }

    rows = []
    for year in years:
        for ws, we in windows(year):
            start = ws.strftime("%Y-%m-%dT00:00:00.000")
            end = we.strftime("%Y-%m-%dT23:59:59.999")
            start_index = 0
            while True:
                params = {
                    "pubStartDate": start,
                    "pubEndDate": end,
                    "resultsPerPage": 2000,
                    "startIndex": start_index,
                }
                for attempt in range(6):
                    try:
                        r = requests.get(base, params=params, headers=headers, timeout=180)
                        if r.status_code == 200:
                            break
                        print(f"  {start[:10]} HTTP {r.status_code}, retrying ...")
                    except Exception as exc:
                        print(f"  {start[:10]} {exc!r}, retrying ...")
                    time.sleep(pause * (attempt + 2))
                else:
                    raise RuntimeError(f"NVD API failed for {start} at {start_index}")
                payload = r.json()
                total = payload.get("totalResults", 0)
                vulns = payload.get("vulnerabilities", [])
                rows.extend(parse(v) for v in vulns)
                start_index += len(vulns)
                if start_index >= total or not vulns:
                    break
                time.sleep(pause)
            print(f"  {start[:10]}..{end[:10]}: {total} CVEs")
            time.sleep(pause)

    df = pd.DataFrame(rows).drop_duplicates("cve_id")
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path} ({len(df):,} rows)")
    return out_path


if any(y <= 2022 for y in EVAL_YEARS) and not NVD_HISTORY.exists():
    fetch_nvd_range((2020, 2021, 2022), NVD_HISTORY)


In [ ]:
def canonical_from_nvd(df, source):
    cwe699 = df["cwe_699_matches"] if "cwe_699_matches" in df.columns else df.get("true_children")
    return pd.DataFrame(
        {
            "cve_id": df["cve_id"].astype(str),
            "year": pd.to_numeric(df["published_year"], errors="coerce"),
            "description": df["description"],
            "all_cwes_raw": df["all_nvd_cwes"],
            "cwe699_raw": cwe699,
            "source": source,
        }
    )


def canonical_from_global(df):
    return pd.DataFrame(
        {
            "cve_id": df["CVE-ID"].astype(str),
            "year": pd.to_numeric(df["YEAR"], errors="coerce"),
            "description": df["CVE-DESCRIPTION"],
            "all_cwes_raw": df["CWE-ID"],
            "cwe699_raw": df["CWE-ID"],
            "source": "global",
        }
    )


nvd_df = hf_read_csv(
    "regularpooria/NVD_CVE_699_Dataset",
    "NVD_CVE_Dataset.csv",
    local_path=NVD_LOCAL,
)
frames = [canonical_from_nvd(nvd_df, "nvd")]

if NVD_HISTORY.exists():
    frames.append(canonical_from_nvd(pd.read_csv(NVD_HISTORY, low_memory=False), "nvd"))
else:
    print("No 2020-2022 NVD cache; the earliest years will use the Global mapping only.")

global_df = hf_read_csv(
    "regularpooria/CVE_CWE_Software_Mapping_Dataset",
    "Global_Dataset.csv",
    local_path=GLOBAL_LOCAL,
)
frames.append(canonical_from_global(global_df))

panel = pd.concat(frames, ignore_index=True)
panel = panel.dropna(subset=["cve_id", "year"])
panel["year"] = panel["year"].astype(int)
panel["_priority"] = panel["source"].map({"nvd": 0, "global": 1}).fillna(2)
panel = (
    panel.sort_values(["cve_id", "_priority"])
    .drop_duplicates("cve_id", keep="first")
    .drop(columns="_priority")
    .reset_index(drop=True)
)
panel["official_all"] = panel["all_cwes_raw"].map(
    lambda v: sorted(set(parse_ids(v)), key=int)
)
panel["official_699"] = panel["cwe699_raw"].map(
    lambda v: sorted(set(parse_ids(v)) & CHILD_ID_SET, key=int)
)

print(f"{len(panel):,} unique CVEs in the panel")
print(panel.groupby("year").size().loc[2019:2026].to_string())


## Exploitation labels and EPSS

**Exploitation labels.** If `scrapers/scraped_data/kev_cross_reference.csv` exists, the
notebook uses it by default: that file is the paper's **three-source union** (CISA KEV,
VulnCheck, GreyNoise) and contains `cisa_exists`/`vc_exists`/`gn_exists` flags plus CISA
and VulnCheck dates. It has 5,036 unique CVEs; 3,085 join the 192k panel, close to the
paper's 3,097. For leakage-safe history, `date_added` is the earlier of the CISA and
VulnCheck dates; every GreyNoise-positive row is also VulnCheck-positive, so this covers
GreyNoise as well.

If the cross-reference file is absent, the notebook falls back to
**CISA KEV + ENISA EUVD KEV** (public). To use a different three-source file instead,
set `USE_KEV_CROSS_REF = False` and point `EXTRA_EXPLOITATION_CSV` at it.

**EPSS.** The FIRST EPSS snapshot is downloaded from Cyentia. It is a *current* snapshot
(2026 in this environment), not the score that was available at disclosure time.
The EPSS comparison is therefore a **retrospective upper bound**, not a deployable
at-disclosure baseline. The paper's commented table makes the same caveat.


In [ ]:
def load_exploitation_labels():
    # Preferred: the paper's three-source cross-reference (CISA + VulnCheck + GreyNoise).
    # Columns: cve_id, cisa_exists, cisa_add_date, vc_exists, vc_add_date, gn_exists.
    if USE_KEV_CROSS_REF and KEV_CROSS_REF.exists():
        cross = pd.read_csv(KEV_CROSS_REF, dtype={"cve_id": str})
        bool_cols = [c for c in ["cisa_exists", "vc_exists", "gn_exists"] if c in cross.columns]
        for c in bool_cols:
            cross[c] = cross[c].astype(str).str.lower().isin(["true", "1", "yes"])
        if bool_cols:
            cross = cross[cross[bool_cols].any(axis=1)].copy()

        cisa_date = pd.to_datetime(cross.get("cisa_add_date"), errors="coerce", utc=True)
        vc_date = pd.to_datetime(cross.get("vc_add_date"), errors="coerce", utc=True)
        # VulnCheck dates exist for every row; CISA dates exist only for CISA-positive
        # rows. The earliest available date is the leakage-safe "known exploited" time.
        cross["date_added"] = pd.concat([cisa_date, vc_date], axis=1).min(axis=1)

        def _sources(row):
            names = []
            if row.get("cisa_exists"):
                names.append("CISA KEV")
            if row.get("vc_exists"):
                names.append("VulnCheck")
            if row.get("gn_exists"):
                names.append("GreyNoise")
            return "|".join(names)

        labels = pd.DataFrame(
            {
                "cve_id": cross["cve_id"].astype(str),
                "date_added": cross["date_added"],
                "label_sources": cross.apply(_sources, axis=1),
                "exploited": 1,
            }
        )
        print(f"Using three-source cross-reference: {KEV_CROSS_REF.name} ({len(labels):,} CVEs)")
        return labels

    frames = []

    kev_path = download(
        "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json",
        CACHE_DIR / "cisa_kev.json",
    )
    with open(kev_path, encoding="utf-8") as f:
        kev = json.load(f)
    kev_df = pd.DataFrame(kev["vulnerabilities"])
    frames.append(
        pd.DataFrame(
            {
                "cve_id": kev_df["cveID"].astype(str),
                "date_added": pd.to_datetime(kev_df["dateAdded"], errors="coerce"),
                "source": "CISA KEV",
            }
        )
    )

    if USE_ENISA_KEV:
        enisa_path = download(
            "https://euvdservices.enisa.europa.eu/api/kev/dump",
            CACHE_DIR / "enisa_kev.json",
        )
        with open(enisa_path, encoding="utf-8") as f:
            enisa = json.load(f)
        enisa_df = pd.DataFrame(enisa)
        if "cveId" in enisa_df.columns:
            frames.append(
                pd.DataFrame(
                    {
                        "cve_id": enisa_df["cveId"].astype(str),
                        "date_added": pd.to_datetime(
                            enisa_df.get("dateAdded"), errors="coerce"
                        ),
                        "source": "ENISA EUVD",
                    }
                )
            )

    if EXTRA_EXPLOITATION_CSV:
        extra = pd.read_csv(EXTRA_EXPLOITATION_CSV)
        cve_col = "cve_id" if "cve_id" in extra.columns else "cve"
        extra_dates = (
            pd.to_datetime(extra["date_added"], errors="coerce")
            if "date_added" in extra.columns
            else pd.NaT
        )
        extra_sources = extra["source"] if "source" in extra.columns else "extra"
        frames.append(
            pd.DataFrame(
                {
                    "cve_id": extra[cve_col].astype(str),
                    "date_added": extra_dates,
                    "source": extra_sources,
                }
            )
        )

    labels = pd.concat(frames, ignore_index=True).dropna(subset=["cve_id"])
    labels["date_added"] = pd.to_datetime(
        labels["date_added"], errors="coerce", utc=True
    )
    labels = labels.groupby("cve_id", as_index=False).agg(
        date_added=("date_added", "min"),
        label_sources=("source", lambda s: "|".join(sorted(set(s.dropna().astype(str))))),
    )
    labels["exploited"] = 1
    return labels


def load_epss():
    epss_path = download(
        "https://epss.cyentia.com/epss_scores-current.csv.gz",
        CACHE_DIR / "epss_scores_current.csv.gz",
    )
    with gzip.open(epss_path, "rt", encoding="utf-8", errors="replace") as f:
        first = f.readline().strip()
    date_match = re.search(r"score_date:([^,]+)", first)
    score_date = date_match.group(1) if date_match else "unknown"
    df = pd.read_csv(epss_path, compression="gzip", comment="#")
    df = df.rename(columns={"cve": "cve_id"})[["cve_id", "epss", "percentile"]]
    df["cve_id"] = df["cve_id"].astype(str)
    return df, score_date


labels = load_exploitation_labels()
panel = panel.merge(
    labels[["cve_id", "date_added", "exploited", "label_sources"]],
    on="cve_id",
    how="left",
)
panel["exploited"] = panel["exploited"].fillna(0).astype(int)

EPSS_DATE = None
if USE_EPSS:
    epss_df, EPSS_DATE = load_epss()
    panel = panel.merge(epss_df[["cve_id", "epss"]], on="cve_id", how="left")

print(f"Exploitation label rows: {len(labels):,}")
print(labels["label_sources"].value_counts().to_string())
if USE_EPSS:
    print(f"EPSS snapshot: {EPSS_DATE}")


## Build the evaluation frame and run HiCWE

Only evaluation-year CVEs with a non-empty description are encoded. If `SAMPLE_SIZE` is
set, each year is sampled independently with a fixed seed. The predictions are cached by
a hash of the CVE ids and the model tag, so changing the sample or the years creates a new
cache entry rather than silently reusing the wrong predictions.

The model is loaded from `models/hf_upload/HiCWE-MiniLM-*` when present; otherwise from
the Hugging Face Hub.


In [ ]:
def build_eval_frame(panel, years, sample_size=None):
    eval_df = panel[panel["year"].isin(years)].copy()
    eval_df = eval_df[eval_df["description"].notna()].reset_index(drop=True)
    if sample_size:
        parts = [
            g.sample(min(sample_size, len(g)), random_state=RANDOM_SEED)
            for _, g in eval_df.groupby("year")
        ]
        eval_df = pd.concat(parts, ignore_index=True)
    return eval_df.reset_index(drop=True)


def resolve_model_path(tag):
    local = BASE_DIR / "models" / "hf_upload" / f"HiCWE-MiniLM-{tag}"
    if (local / "model.safetensors").exists() or (local / "pytorch_model.bin").exists():
        return str(local)
    return f"regularpooria/HiCWE-MiniLM-{tag}"


def predict_hicwe(df, tag=MODEL_TAG, k=KEEP_TOPK):
    cache_key = hashlib.md5(
        (
            "|".join(sorted(df["cve_id"].astype(str)))
            + f"|{tag}|{k}|{TAXONOMY_HASH}"
        ).encode()
    ).hexdigest()[:12]
    cache_path = CACHE_DIR / f"hicwe_{tag}_{cache_key}.csv"
    if cache_path.exists():
        print(f"Loading cached predictions: {cache_path.name}")
        return pd.read_csv(cache_path, dtype={"cve_id": str})

    model_path = resolve_model_path(tag)
    print(f"Loading {model_path} on {DEVICE} ...")
    model = SentenceTransformer(model_path, device=DEVICE)
    model.max_seq_length = 512  # matches the paper's training configuration

    child_cache = CACHE_DIR / f"hicwe_{tag}_child_embeddings_{TAXONOMY_HASH}.npy"
    if child_cache.exists():
        child_emb = np.load(child_cache)
    else:
        print("Embedding CWE-699 child nodes ...")
        child_emb = model.encode(
            CHILD_TEXTS,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)
        np.save(child_cache, child_emb)

    texts = df["description"].fillna("").astype(str).tolist()
    all_top = []
    all_scores = []
    chunk = 2048
    t0 = time.time()
    for start in range(0, len(texts), chunk):
        emb = model.encode(
            texts[start : start + chunk],
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)
        sims = emb @ child_emb.T
        part = np.argpartition(-sims, k - 1, axis=1)[:, :k]
        part_scores = np.take_along_axis(sims, part, axis=1)
        order = np.argsort(-part_scores, axis=1)
        all_top.append(np.take_along_axis(part, order, axis=1))
        all_scores.append(np.take_along_axis(part_scores, order, axis=1))
        print(
            f"  encoded {min(start + chunk, len(texts)):,}/{len(texts):,} CVEs "
            f"({time.time() - t0:.0f}s)"
        )

    top = np.vstack(all_top)
    top_scores = np.vstack(all_scores)
    out = pd.DataFrame({"cve_id": df["cve_id"].astype(str).values})
    for j in range(k):
        out[f"hicwe_top{j+1}"] = [CHILD_IDS[i] for i in top[:, j]]
        out[f"hicwe_top{j+1}_score"] = top_scores[:, j]
    out.to_csv(cache_path, index=False)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out


eval_df = build_eval_frame(panel, EVAL_YEARS, SAMPLE_SIZE)
print(f"Evaluation CVEs: {len(eval_df):,}")
preds = predict_hicwe(eval_df, MODEL_TAG, KEEP_TOPK)
eval_df = eval_df.merge(preds, on="cve_id", how="left")

summary = (
    eval_df.assign(
        has_official_699=eval_df["official_699"].map(len).gt(0),
        has_epss=eval_df["epss"].notna() if "epss" in eval_df else False,
    )
    .groupby("year")
    .agg(
        cves=("cve_id", "size"),
        exploited=("exploited", "sum"),
        official699=("has_official_699", "sum"),
        epss=("has_epss", "sum"),
    )
)
summary["prevalence"] = summary["exploited"] / summary["cves"]
display(summary)


## Sanity check: does this panel reproduce the paper's CWE accuracy?

Before interpreting the downstream ranking, confirm that the encoder is doing its job on
this exact evaluation frame. `top-k any` counts a hit if any of the top-k predictions is
in the official CWE-699 label set; `top-1 primary` requires the first official label to be
the top-1 prediction. On the full 2023–2025 panel, `top-1 any` should be close to the
paper's **0.789**; on a small sample it will be noisy.


In [ ]:
def cwe_accuracy_table(df):
    rows = []
    for year, g in df.groupby("year"):
        g = g[g["official_699"].map(len).gt(0)]
        if g.empty:
            continue
        off = g["official_699"].tolist()
        top1 = g["hicwe_top1"].astype(str).tolist()
        top3 = g[["hicwe_top1", "hicwe_top2", "hicwe_top3"]].astype(str).to_numpy()
        top5 = g[
            ["hicwe_top1", "hicwe_top2", "hicwe_top3", "hicwe_top4", "hicwe_top5"]
        ].astype(str).to_numpy()
        primary = [o[0] if o else None for o in off]
        rows.append(
            {
                "year": int(year),
                "n": len(g),
                "top1_any": float(np.mean([a in o for a, o in zip(top1, off)])),
                "top3_any": float(np.mean([bool(set(r) & set(o)) for r, o in zip(top3, off)])),
                "top5_any": float(np.mean([bool(set(r) & set(o)) for r, o in zip(top5, off)])),
                "top1_primary": float(np.mean([a == p for a, p in zip(top1, primary)])),
            }
        )
    return pd.DataFrame(rows)


accuracy_df = cwe_accuracy_table(eval_df)
print("HiCWE label accuracy on the evaluation frame (official699_matched CVEs)")
display(accuracy_df.round(4))


## Where does the prioritization signal go?

Overall accuracy is not the quantity that matters downstream. A model can be 80% accurate
and still lose the ranking signal if its errors concentrate on the vulnerabilities that
are actually exploited. The first table below splits top-1 accuracy into the **exploited**
and **not-exploited** subsets. The second drills into the worst year (2022 by default) and
shows which true CWEs the errors come from.

This is the diagnostic that explains the 2022 exception in the ranking results.


In [ ]:
rows = []
for year, g in eval_df.groupby("year"):
    g = g[g["official_699"].map(len).gt(0)].copy()
    g["_hit"] = [p in truth for p, truth in zip(g["hicwe_top1"], g["official_699"])]
    rows.append(
        {
            "year": int(year),
            "n": len(g),
            "exploited": int(g["exploited"].sum()),
            "top1_all": float(g["_hit"].mean()),
            "top1_exploited": float(g.loc[g["exploited"] == 1, "_hit"].mean()),
            "top1_not_exploited": float(g.loc[g["exploited"] == 0, "_hit"].mean()),
        }
    )
subset_accuracy = pd.DataFrame(rows)
subset_accuracy["exploited_minus_clean"] = (
    subset_accuracy["top1_exploited"] - subset_accuracy["top1_not_exploited"]
)
print("Top-1 accuracy on the exploited vs. not-exploited subsets")
display(subset_accuracy.round(4))

FOCUS_YEAR = 2022
g = eval_df[
    (eval_df["year"] == FOCUS_YEAR)
    & (eval_df["official_699"].map(len).gt(0))
    & (eval_df["exploited"] == 1)
].copy()
g["true_primary"] = g["official_699"].map(lambda x: x[0] if x else None)
g["_hit"] = [p in truth for p, truth in zip(g["hicwe_top1"], g["official_699"])]
per_cwe = (
    g.groupby("true_primary")
    .agg(n=("cve_id", "size"), top1_hit=("_hit", "mean"))
    .query("n >= 3")
    .sort_values(["n", "top1_hit"], ascending=[False, True])
)
print(f"\nPer-CWE top-1 hit rate for exploited CVEs in {FOCUS_YEAR} (n >= 3)")
display(per_cwe.round(3))
print(
    "If a few high-risk CWEs have very low hit rates, that is the mechanism behind the "
    "year's ranking loss -- not overall accuracy."
)


## Ranking metrics and the historical score

`expected_ranking_metrics` computes the exact expectation of P@k, R@k and NDCG@k under
uniform random tie-breaking. If a score group has `n_g` CVEs and `e_g` positives, every
position in that group is positive with probability `e_g / n_g`; summing those
probabilities gives the expected metric without 1,000 Monte-Carlo permutations.

`history_scores` implements Eq. (5). It builds a long `(CVE, CWE)` table from years
before the evaluation year, counts `E_c` and `N_c`, and smooths with `alpha * P0`.
`score_with_labels` / `score_with_topk` implement Eq. (6), the max over a CVE's labels.

`HISTORY_MODE="publication"` counts a historical CVE as exploited if it is exploited in
the final snapshot (the paper's setup). `"knowledge"` only counts it if its KEV/ENISA
`date_added` is before the evaluation year, which is the leakage-safe variant; the
notebook reports both as a sensitivity check.


In [ ]:
def expected_ranking_metrics(scores, labels, ks):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=bool)
    n = len(labels)
    positives = int(labels.sum())
    if n == 0:
        return {f"{m}{k}": np.nan for m in ("p", "r", "ndcg") for k in ks}

    order = np.argsort(-scores, kind="mergesort")
    s_sorted = scores[order]
    y_sorted = labels[order]

    change = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    starts = np.flatnonzero(change)
    ends = np.r_[starts[1:], n]
    group_n = (ends - starts).astype(float)
    group_e = np.add.reduceat(y_sorted.astype(float), starts)
    group_p = group_e / group_n

    inv_log = 1.0 / np.log2(np.arange(2, n + 2))
    pref_inv_log = np.r_[0.0, np.cumsum(inv_log)]

    cum_n_before = np.r_[0.0, np.cumsum(group_n)[:-1]]
    cum_e_before = np.r_[0.0, np.cumsum(group_e)[:-1]]
    cum_dcg_before = np.r_[
        0.0, np.cumsum(group_p * np.add.reduceat(inv_log, starts))[:-1]
    ]

    out = {}
    for k in ks:
        kk = min(int(k), n)
        g = int(np.searchsorted(cum_n_before, kk, side="right") - 1)
        start = cum_n_before[g]
        take = min(kk - start, group_n[g])
        exp_pos = cum_e_before[g] + take * group_p[g]
        dcg = cum_dcg_before[g] + group_p[g] * (
            pref_inv_log[int(start + take)] - pref_inv_log[int(start)]
        )
        idcg = pref_inv_log[min(kk, positives)]
        out[f"p{k}"] = exp_pos / kk
        out[f"r{k}"] = exp_pos / positives if positives else 0.0
        out[f"ndcg{k}"] = dcg / idcg if idcg > 0 else 0.0
    return out


def evaluate_scores(y, scores, ks):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.sum() == 0 or y.sum() == len(y):
        pr_auc = float(y.mean())
        roc_auc = np.nan
    else:
        pr_auc = float(average_precision_score(y, scores))
        roc_auc = float(roc_auc_score(y, scores))
    metrics = {
        "n": len(y),
        "positives": int(y.sum()),
        "prevalence": float(y.mean()) if len(y) else np.nan,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
    }
    metrics.update(expected_ranking_metrics(scores, y, ks))
    return metrics


def history_scores(panel, label_col, cutoff_year, alpha=ALPHA, mode=HISTORY_MODE):
    history = panel[panel["year"] < cutoff_year].copy()
    if history.empty:
        return {}, np.nan, 0

    if mode == "knowledge":
        cutoff = pd.Timestamp(year=int(cutoff_year), month=1, day=1, tz="UTC")
        exploited = history["date_added"].notna() & (history["date_added"] < cutoff)
    else:
        exploited = history["exploited"].astype(bool)
    history["_exploited"] = exploited.astype(int)

    # P0 is the global historical rate over all CVEs, including CVEs with no
    # usable label. E_c and N_c are counted only for labeled CVE-CWE pairs.
    p0 = float(history["_exploited"].mean())
    labeled = history[history[label_col].map(len) > 0]
    if labeled.empty:
        return {}, p0, 0

    long = labeled[["cve_id", "_exploited", label_col]].explode(label_col)
    long = long.rename(columns={label_col: "cwe"})
    grp = long.groupby("cwe")["_exploited"].agg(["sum", "count"])
    scores = (grp["sum"] + alpha * p0) / (grp["count"] + alpha)
    return scores.to_dict(), p0, len(labeled)


def score_with_labels(df, label_col, score_map, p0):
    out = np.empty(len(df), dtype=float)
    for i, labels in enumerate(df[label_col].to_numpy()):
        vals = [score_map.get(str(c), p0) for c in labels]
        out[i] = max(vals) if vals else p0
    return out


def score_with_topk(df, cols, score_map, p0):
    arr = df[cols].to_numpy()
    out = np.empty(len(df), dtype=float)
    for i, labels in enumerate(arr):
        vals = [score_map.get(str(c), p0) for c in labels if str(c) != "nan"]
        out[i] = max(vals) if vals else p0
    return out


## Run the simulation

Three cohorts are evaluated:

- **`full`** — every evaluation CVE with a description. This is the deployment coverage
  view; official-CWE methods fall back to `P0` when NVD has no usable label, while HiCWE
  always returns a CWE.
- **`official699_matched`** — CVEs with at least one official CWE-699 label. This is the
  **primary** cohort: official and predicted labels are evaluated on exactly the same
  vulnerabilities.
- **`epss_matched`** — CVEs with an EPSS score. Used only for the EPSS comparison, with
  every method evaluated on the same rows.

`random` is a constant score, which is the expectation of a uniformly random permutation;
the analytic tie-breaking then yields the same prevalence-based P@k/R@k/NDCG as the
paper's 1,000-shuffle baseline.


In [ ]:
score_store = {}
rows = []
rng = np.random.default_rng(RANDOM_SEED)

for year in EVAL_YEARS:
    test = eval_df[eval_df["year"] == year].reset_index(drop=True)
    if test.empty:
        continue

    hist_all, p0_all, n_hist_all = history_scores(panel, "official_all", year)
    hist_699, p0_699, n_hist_699 = history_scores(panel, "official_699", year)

    primary_col = test["official_699"].map(lambda x: x[:1] if isinstance(x, list) else [])
    test["official_699_primary"] = primary_col

    base_scores = {
        "official_all": score_with_labels(test, "official_all", hist_all, p0_all),
        "official_699": score_with_labels(test, "official_699", hist_699, p0_699),
        "official_699_primary": score_with_labels(
            test, "official_699_primary", hist_699, p0_699
        ),
        "hicwe_top1": score_with_topk(test, ["hicwe_top1"], hist_699, p0_699),
        "hicwe_top3": score_with_topk(
            test, ["hicwe_top1", "hicwe_top2", "hicwe_top3"], hist_699, p0_699
        ),
        "hicwe_top5": score_with_topk(
            test,
            ["hicwe_top1", "hicwe_top2", "hicwe_top3", "hicwe_top4", "hicwe_top5"],
            hist_699,
            p0_699,
        ),
        "no_cwe": np.full(len(test), p0_699, dtype=float),
        "random": np.zeros(len(test), dtype=float),
    }
    if USE_EPSS:
        base_scores["epss"] = test["epss"].fillna(0.0).to_numpy()

    coverage = {
        "official_all": test["official_all"].map(len).gt(0).to_numpy(),
        "official_699": test["official_699"].map(len).gt(0).to_numpy(),
        "official_699_primary": test["official_699"].map(len).gt(0).to_numpy(),
        "hicwe_top1": np.ones(len(test), dtype=bool),
        "hicwe_top3": np.ones(len(test), dtype=bool),
        "hicwe_top5": np.ones(len(test), dtype=bool),
        "no_cwe": np.ones(len(test), dtype=bool),
        "random": np.ones(len(test), dtype=bool),
        "epss": test["epss"].notna().to_numpy() if USE_EPSS else np.ones(len(test), dtype=bool),
    }

    cohorts = {
        "full": np.ones(len(test), dtype=bool),
        "official699_matched": test["official_699"].map(len).gt(0).to_numpy(),
    }
    if USE_EPSS:
        cohorts["epss_matched"] = test["epss"].notna().to_numpy()

    for cohort_name, mask in cohorts.items():
        if mask.sum() == 0:
            continue
        for method, scores in base_scores.items():
            if cohort_name != "epss_matched" and method == "epss":
                continue
            if cohort_name == "epss_matched" and method not in {
                "official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5",
                "no_cwe", "random", "epss",
            }:
                continue
            sub = test[mask]
            metrics = evaluate_scores(sub["exploited"].to_numpy(), scores[mask], KS)
            rows.append(
                {
                    "year": year,
                    "cohort": cohort_name,
                    "method": method,
                    "coverage": float(coverage[method][mask].mean()),
                    **metrics,
                }
            )
            score_store[(year, cohort_name, method)] = (
                scores[mask],
                sub["exploited"].to_numpy(),
                sub["cve_id"].to_numpy(),
            )

    print(
        f"{year}: {len(test):,} CVEs, {int(test['exploited'].sum())} exploited, "
        f"{int(test['official_699'].map(len).gt(0).sum()):,} with official CWE-699; "
        f"history={n_hist_699:,}"
    )

results = pd.DataFrame(rows)
print(f"\n{len(results):,} result rows")


## Why top-3/top-5 candidate sets do not help

The simulation treats top-$k$ predictions as a multi-label set and takes the maximum
historical score, following Eq. (6). That is the natural extension, but it is not a
calibrated multi-label distribution: adding candidates can only increase a CVE's score,
so many non-exploited CVEs acquire a high score, the number of distinct score levels
falls, and a large mass of CVEs ties at the maximum. The diagnostic below shows this
directly. It is the mechanism behind the lower PR-AUC/P@100 retention for top-3 and
top-5 in the retention table.


In [ ]:
diag_rows = []
for year in EVAL_YEARS:
    for method in ["official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5"]:
        key = (year, "official699_matched", method)
        if key not in score_store:
            continue
        scores = score_store[key][0]
        diag_rows.append(
            {
                "year": year,
                "method": method,
                "unique_scores": int(len(np.unique(scores))),
                "share_at_max": float((scores == scores.max()).mean()),
                "mean_score": float(scores.mean()),
            }
        )
topk_diag = pd.DataFrame(diag_rows)
display(
    topk_diag.pivot_table(
        index="year",
        columns="method",
        values=["unique_scores", "share_at_max", "mean_score"],
    ).round(4)
)
print(
    "Top-1 has a score distribution close to the official labels; top-3/top-5 collapse "
    "many CVEs onto a few high scores. For deployment, report top-1 as the primary mode."
)


## Main results

The primary table is `official699_matched`. `official_699` is the label-space-matched
ground-truth ranking; `hicwe_top1` is the same pipeline with one predicted label.
`official_all` is the paper's empirical-study label space and is shown for reference.

PR-AUC is the primary metric. P@100 is the practically relevant "what is at the top of
the remediation queue" metric.


In [ ]:
matched = results[results["cohort"] == "official699_matched"].copy()
methods_order = [
    "official_all", "official_699", "official_699_primary",
    "hicwe_top1", "hicwe_top3", "hicwe_top5",
    "no_cwe", "random",
]

pr = matched.pivot_table(index="year", columns="method", values="pr_auc").reindex(
    columns=methods_order
)
p100 = matched.pivot_table(index="year", columns="method", values="p100").reindex(
    columns=methods_order
)
roc = matched.pivot_table(index="year", columns="method", values="roc_auc").reindex(
    columns=methods_order
)

print("PR-AUC — official699_matched cohort")
display(pr.round(4))
print("\nPrecision@100 — official699_matched cohort")
display(p100.round(4))
print("\nROC-AUC — official699_matched cohort")
display(roc.round(4))

full = results[results["cohort"] == "full"]
full_methods = ["official_all", "official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5", "no_cwe"]
full_pr = full.pivot_table(index="year", columns="method", values="pr_auc").reindex(
    columns=full_methods
)
print("\nPR-AUC — full cohort (deployment coverage; official methods fall back to P0)")
display(full_pr.round(4))

full_cov = full.pivot_table(index="year", columns="method", values="coverage").reindex(
    columns=full_methods
)
print("\nCoverage — full cohort (fraction of CVEs with a usable label/score)")
display(full_cov.round(4))


## Signal retention

**Signal retention** is the ratio of the HiCWE ranking metric to the official-CWE metric
on the same CVEs. A value of 1.0 means predicted labels preserve the entire ranking
signal; 0.8 means they retain 80%; values below ~0.7 would mean predicted labels are not
a drop-in replacement for the prioritization use case.

The table below reports both years 2021–2025 (to match the paper's table) and the
deployment-relevant 2023–2025 test window separately. 2021 is partly in HiCWE's training
period, so 2023–2025 is the honest end-to-end test.


In [ ]:
def retention_table(results, cohort="official699_matched"):
    sub = results[results["cohort"] == cohort]
    methods = ["official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5"]
    wide = sub[sub["method"].isin(methods)].pivot_table(
        index="year", columns="method", values=["pr_auc", "p100"]
    )
    out_rows = []
    for year in wide.index:
        row = {"year": int(year)}
        for metric in ["pr_auc", "p100"]:
            official = wide.loc[year, (metric, "official_699")]
            row[f"{metric}_official_699"] = official
            for method in ["hicwe_top1", "hicwe_top3", "hicwe_top5"]:
                value = wide.loc[year, (metric, method)]
                row[f"{metric}_{method}"] = value
                row[f"{metric}_{method}_ratio"] = value / official if official else np.nan
                row[f"{metric}_{method}_delta"] = value - official
        out_rows.append(row)
    return pd.DataFrame(out_rows)


ret = retention_table(results)
print("Signal retention — official699_matched cohort")
display(
    ret[
        [
            "year",
            "pr_auc_official_699",
            "pr_auc_hicwe_top1",
            "pr_auc_hicwe_top1_ratio",
            "pr_auc_hicwe_top3_ratio",
            "pr_auc_hicwe_top5_ratio",
            "p100_hicwe_top1_ratio",
            "p100_hicwe_top3_ratio",
            "p100_hicwe_top5_ratio",
        ]
    ].rename(
        columns={
            "pr_auc_official_699": "PR-AUC official",
            "pr_auc_hicwe_top1": "PR-AUC HiCWE top-1",
            "pr_auc_hicwe_top1_ratio": "PR-AUC top1 / official",
            "pr_auc_hicwe_top3_ratio": "PR-AUC top3 / official",
            "pr_auc_hicwe_top5_ratio": "PR-AUC top5 / official",
            "p100_hicwe_top1_ratio": "P@100 top1 / official",
            "p100_hicwe_top3_ratio": "P@100 top3 / official",
            "p100_hicwe_top5_ratio": "P@100 top5 / official",
        }
    ).round(4)
)

deploy_years = [y for y in [2023, 2024, 2025] if y in set(ret["year"])]
paper_years = [y for y in EVAL_YEARS if y in set(ret["year"])]
for label, years in [("2023-2025 (deployment-relevant)", deploy_years),
                     ("2021-2025 (paper window)", paper_years)]:
    if not years:
        continue
    block = ret[ret["year"].isin(years)]
    print(f"\n{label} macro averages:")
    display(
        block[
            [
                "pr_auc_hicwe_top1_ratio", "pr_auc_hicwe_top3_ratio",
                "pr_auc_hicwe_top5_ratio", "p100_hicwe_top1_ratio",
                "p100_hicwe_top3_ratio", "p100_hicwe_top5_ratio",
            ]
        ].mean().rename(
            {
                "pr_auc_hicwe_top1_ratio": "PR-AUC top1 / official",
                "pr_auc_hicwe_top3_ratio": "PR-AUC top3 / official",
                "pr_auc_hicwe_top5_ratio": "PR-AUC top5 / official",
                "p100_hicwe_top1_ratio": "P@100 top1 / official",
                "p100_hicwe_top3_ratio": "P@100 top3 / official",
                "p100_hicwe_top5_ratio": "P@100 top5 / official",
            }
        ).to_frame("ratio")
    )


## EPSS comparison

EPSS is the strongest baseline, but the snapshot is **retrospective** (2026 scores used to
rank 2021–2025 CVEs). It therefore has an information advantage and should be read as a
ceiling, not as a like-for-like deployable baseline. The point of the table is to show the
gap between weakness-history-only ranking and a mature, feature-rich exploitation model,
and to make clear that the CWE signal is complementary.

Every method in this table is evaluated on the same `epss_matched` CVEs.


In [ ]:
if USE_EPSS:
    epss_view = results[results["cohort"] == "epss_matched"]
    epss_methods = ["official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5", "epss", "random"]
    epss_pr = epss_view.pivot_table(index="year", columns="method", values="pr_auc").reindex(
        columns=epss_methods
    )
    epss_p100 = epss_view.pivot_table(index="year", columns="method", values="p100").reindex(
        columns=epss_methods
    )
    print(f"EPSS snapshot: {EPSS_DATE} (retrospective; see caveat above)")
    print("\nPR-AUC — epss_matched cohort")
    display(epss_pr.round(4))
    print("\nPrecision@100 — epss_matched cohort")
    display(epss_p100.round(4))
else:
    print("USE_EPSS=False; skipping the EPSS comparison.")


## Paired bootstrap: is the HiCWE–official difference real?

The bootstrap resamples CVEs within each year and recomputes the PR-AUC difference
`hicwe_top1 - official_699` on the matched cohort. The interval is a 95% percentile
interval. With only a few hundred exploited CVEs per year, the intervals are wide; that
is itself an important result to report.


In [ ]:
def paired_bootstrap(y, score_a, score_b, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    diffs = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y[idx]
        if yy.sum() == 0 or yy.sum() == n:
            continue
        diffs.append(
            average_precision_score(yy, score_a[idx])
            - average_precision_score(yy, score_b[idx])
        )
    if not diffs:
        return np.nan, np.nan, np.nan
    return (
        float(np.mean(diffs)),
        float(np.percentile(diffs, 2.5)),
        float(np.percentile(diffs, 97.5)),
    )


boot_rows = []
for year in EVAL_YEARS:
    key_a = (year, "official699_matched", "hicwe_top1")
    key_b = (year, "official699_matched", "official_699")
    if key_a not in score_store or key_b not in score_store:
        continue
    scores_a, y, _ = score_store[key_a]
    scores_b, _, _ = score_store[key_b]
    mean, lo, hi = paired_bootstrap(y, scores_a, scores_b)
    boot_rows.append(
        {
            "year": year,
            "delta_pr_auc_top1_minus_official": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n": len(y),
            "positives": int(y.sum()),
        }
    )

bootstrap_df = pd.DataFrame(boot_rows)
display(bootstrap_df.round(4))


## Alpha sensitivity and leakage-safe history

**Alpha.** The paper selected `alpha=40` by sweeping `{20, 40, 60}`. To avoid tuning on the
evaluation years, the notebook re-runs the selection on **2020**, a year before the
2021–2025 evaluation window. If 2020 is available in the panel, this gives a clean
validation check. If it is not, the cell prints a note.

**History leakage.** The paper's historical counts use final exploitation labels. That can
leak future exploitation knowledge into the historical rate. The leakage-safe variant only
counts a historical CVE as exploited if its KEV/ENISA `date_added` precedes the evaluation
year. The table shows whether the conclusions depend on the difference.


In [ ]:
if 2020 in set(panel["year"]):
    alpha_rows = []
    test_2020 = panel[(panel["year"] == 2020)].copy()
    test_2020 = test_2020[
        test_2020["description"].notna() & test_2020["official_699"].map(len).gt(0)
    ]
    for alpha in [20, 40, 60]:
        hist, p0, _ = history_scores(
            panel, "official_699", 2020, alpha=float(alpha), mode="publication"
        )
        scores = score_with_labels(test_2020, "official_699", hist, p0)
        m = evaluate_scores(test_2020["exploited"].to_numpy(), scores, [100])
        alpha_rows.append(
            {
                "validation_year": 2020,
                "alpha": alpha,
                "n": m["n"],
                "positives": m["positives"],
                "pr_auc": m["pr_auc"],
                "roc_auc": m["roc_auc"],
                "p100": m["p100"],
            }
        )
    alpha_df = pd.DataFrame(alpha_rows)
    print("Alpha selection on 2020 (before the evaluation window):")
    display(alpha_df.round(4))
else:
    alpha_df = pd.DataFrame()
    print("2020 is not available in the panel; skipping the alpha validation check.")

leak_rows = []
for year in [y for y in [2023, 2024, 2025] if y in set(eval_df["year"])]:
    test = eval_df[eval_df["year"] == year]
    test = test[test["official_699"].map(len).gt(0)]
    if test.empty:
        continue
    for mode in ["publication", "knowledge"]:
        hist, p0, _ = history_scores(panel, "official_699", year, mode=mode)
        off = score_with_labels(test, "official_699", hist, p0)
        hi = score_with_topk(test, ["hicwe_top1"], hist, p0)
        m_off = evaluate_scores(test["exploited"].to_numpy(), off, [100])
        m_hi = evaluate_scores(test["exploited"].to_numpy(), hi, [100])
        leak_rows.append(
            {
                "year": year,
                "history_mode": mode,
                "official_699_pr_auc": m_off["pr_auc"],
                "hicwe_top1_pr_auc": m_hi["pr_auc"],
                "official_699_p100": m_off["p100"],
                "hicwe_top1_p100": m_hi["p100"],
            }
        )
leak_df = pd.DataFrame(leak_rows)
if not leak_df.empty:
    print("\nHistory-label sensitivity (publication vs. knowledge cutoff):")
    display(leak_df.round(4))


## Figures

1. PR-AUC by year: official CWE vs. HiCWE top-1/3/5 vs. random.
2. Precision@100 by year.
3. Signal-retention ratios.
4. Precision–recall curves for the latest evaluation year (on the matched cohort) and for
   EPSS on its matched cohort.


In [ ]:
import matplotlib.pyplot as plt

fig_dir = OUT_DIR
matched = results[results["cohort"] == "official699_matched"]
methods_plot = ["official_699", "hicwe_top1", "hicwe_top3", "hicwe_top5", "random"]
colors = {
    "official_699": "#111111",
    "hicwe_top1": "#1f77b4",
    "hicwe_top3": "#2ca02c",
    "hicwe_top5": "#9467bd",
    "random": "#bbbbbb",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, metric, title in [
    (axes[0], "pr_auc", "PR-AUC"),
    (axes[1], "p100", "Precision@100"),
]:
    for method in methods_plot:
        sub = matched[matched["method"] == method].sort_values("year")
        if sub.empty:
            continue
        ax.plot(
            sub["year"], sub[metric], marker="o", linewidth=2,
            label=method, color=colors.get(method),
        )
    ax.set_title(title)
    ax.set_xlabel("Evaluation year")
    ax.set_ylabel(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Historical CWE exploitation ranking on official699_matched CVEs")
fig.tight_layout()
fig.savefig(fig_dir / "pr_auc_and_p100_by_year.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
for col, label in [
    ("pr_auc_hicwe_top1_ratio", "PR-AUC top-1 / official"),
    ("pr_auc_hicwe_top3_ratio", "PR-AUC top-3 / official"),
    ("pr_auc_hicwe_top5_ratio", "PR-AUC top-5 / official"),
    ("p100_hicwe_top1_ratio", "P@100 top-1 / official"),
]:
    ax.plot(ret["year"], ret[col], marker="o", linewidth=2, label=label)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_ylim(0, max(1.6, float(ret[[c for c in ret.columns if c.endswith("_ratio")]].max().max())))
ax.set_xlabel("Evaluation year")
ax.set_ylabel("Ratio to official CWE")
ax.set_title("Signal retention: HiCWE vs. official-CWE ranking")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / "signal_retention.png", dpi=180)
plt.show()

# PR curves for the latest year
from sklearn.metrics import precision_recall_curve

latest = max(EVAL_YEARS)
fig, ax = plt.subplots(figsize=(6.5, 5))
for method in ["official_699", "hicwe_top1", "hicwe_top3"]:
    key = (latest, "official699_matched", method)
    if key not in score_store:
        continue
    scores, y, _ = score_store[key]
    precision, recall, _ = precision_recall_curve(y, scores)
    ax.plot(recall, precision, linewidth=2, label=f"{method} (AP={average_precision_score(y, scores):.3f})")

if USE_EPSS and (latest, "epss_matched", "epss") in score_store:
    scores, y, _ = score_store[(latest, "epss_matched", "epss")]
    precision, recall, _ = precision_recall_curve(y, scores)
    ax.plot(recall, precision, linewidth=2, linestyle="--",
            label=f"EPSS retrospective (AP={average_precision_score(y, scores):.3f})")

base = float(np.mean([y.mean() for (yr, coh, m), (s, y, ids) in score_store.items()
                      if yr == latest and coh == "official699_matched" and m == "official_699"]))
ax.axhline(base, color="grey", linestyle=":", label=f"prevalence={base:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision–recall, {latest} (matched cohort)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / f"pr_curve_{latest}.png", dpi=180)
plt.show()


## Save the outputs

All tables and figures are written to `experiments/simulation_outputs/`. The prediction
file is the one to keep: it contains the official CWE-699 labels, HiCWE top-5 predictions,
the exploitation outcome, and the EPSS score for every evaluated CVE, so the ranking can
be recomputed without re-running the encoder.


In [ ]:
results.to_csv(OUT_DIR / "simulation_results.csv", index=False)
ret.to_csv(OUT_DIR / "signal_retention.csv", index=False)
accuracy_df.to_csv(OUT_DIR / "cwe_accuracy.csv", index=False)
bootstrap_df.to_csv(OUT_DIR / "bootstrap_ci.csv", index=False)
if USE_EPSS:
    results[results["cohort"] == "epss_matched"].to_csv(
        OUT_DIR / "epss_comparison.csv", index=False
    )
if not alpha_df.empty:
    alpha_df.to_csv(OUT_DIR / "alpha_sensitivity.csv", index=False)
if not leak_df.empty:
    leak_df.to_csv(OUT_DIR / "leakage_safe_history.csv", index=False)

pred_cols = [
    "cve_id", "year", "official_all", "official_699",
    "hicwe_top1", "hicwe_top2", "hicwe_top3", "hicwe_top4", "hicwe_top5",
    "hicwe_top1_score", "hicwe_top2_score", "hicwe_top3_score",
    "hicwe_top4_score", "hicwe_top5_score",
    "exploited", "epss",
]
pred_cols = [c for c in pred_cols if c in eval_df.columns]
export = eval_df[pred_cols].copy()
for c in ["official_all", "official_699"]:
    export[c] = export[c].map(lambda x: "|".join(map(str, x)) if isinstance(x, list) else "")
export.to_csv(OUT_DIR / "cve_predictions_and_labels.csv", index=False)
summary.to_csv(OUT_DIR / "panel_summary.csv")

print("Wrote:")
for path in sorted(OUT_DIR.glob("*")):
    print(" ", path.name)


## How to read the result

The notebook is designed to answer one question: **does replacing official NVD CWE labels
with HiCWE predictions preserve the historical-exploitation ranking signal?**

Use the `2023–2025` macro averages in the signal-retention table:

- **Retention near 1.0** — predicted labels are a drop-in replacement for this
  prioritization use case. The end-to-end claim is supported.
- **Retention ~0.7–0.9** — the signal is useful but degraded; predicted labels should be
  described as a complementary, lower-confidence signal, not as equivalent to NVD labels.
- **Retention below ~0.7** — the accuracy gain of HiCWE does not translate into the
  downstream prioritization task; the paper should not claim that predicted CWEs *support*
  prioritization without this caveat.

The `full` cohort is the deployment-coverage view: HiCWE always returns a CWE, so it can
rank CVEs that NVD has not labeled. That is a genuine advantage, but it is a different
question from label fidelity. Report both.

### Honest caveats to carry into the paper

1. **Exploitation labels and coverage.** If `kev_cross_reference.csv` is present, the run
   uses the paper's three-source union (CISA + VulnCheck + GreyNoise); otherwise it falls
   back to public CISA + ENISA. The three-source labels give absolute PR-AUC values close
   to the paper's Finding 6 table. However, the CWE-699 matched cohort covers only 56--67\%
   of all exploited CVEs (2021: 277/476; 2022: 302/452; 2023: 318/534; 2024: 454/677;
   2025: 352/631), so the simulation applies to the CWE-699 label space, not to every
   exploited CVE. State that scope explicitly.
2. **EPSS is retrospective.** The snapshot is from 2026, after all evaluation years. It
   should be labelled a ceiling, not a deployable baseline.
3. **Matched cohort is primary.** The 2023–2025 NVD file published in this repo is already
   restricted to CVEs with a CWE-699 match. The notebook therefore treats
   `official699_matched` as the main cohort. This is the cleanest test of prediction
   fidelity, but it is not the paper's full 29k/40k/50k yearly cohorts. If you want the
   full coverage view, fetch the unfiltered NVD years with the same fetcher.
4. **Top-k is not calibrated multi-label output.** Treating top-3/top-5 candidates as a
   label set is optimistic: it gives HiCWE several chances to hit a high-risk CWE. Top-1 is
   the honest deployment number.
5. **Single model, single seed.** The notebook evaluates one HiCWE checkpoint per run.
   Repeat with `MODEL_TAG="L6"` and `"L12"` to report both, as the paper does.

### What to add to the paper

A short subsection after the current Finding 6 table:

> *End-to-end simulation.* Replacing NVD CWE assignments with HiCWE top-1 predictions on
> the 2023–2025 matched cohort preserves **X%** of the official-CWE PR-AUC and **Y%** of
> Precision@100. Top-3/top-5 candidate sets recover **Z%** / **W%**. The degradation is
> concentrated in [describe: tail CWEs, multi-label CVEs, or specific categories]. EPSS
> remains substantially stronger on the retrospective snapshot, confirming that historical
> weakness-level exploitation is a complementary signal rather than a replacement for
> vulnerability-specific exploit prediction.

Fill in the X/Y/Z/W from the `2023-2025` macro-average block. That single paragraph is the
experiment the reviewer is asking for, and it turns the paper's two halves into one
empirical claim.
